# Brazilian E-Commerce Analysis — Olist

**A business analysis of sales, customers, sellers and delivery performance using Python, SQL and visualisation.**

> **Notebook guide:** This notebook contains the full analysis and explanations. The repository `README.md` is a concise landing page with the dataset source, tools, headline findings and file guide; it should not repeat the notebook section by section.

## Analysis roadmap
1. Load and inspect the nine source tables
2. Assess data quality and prepare analytical fields
3. Build a relational SQLite database
4. Answer business questions with SQL
5. Explore sales, customer and delivery patterns
6. Summarise findings, recommendations and limitations

**Dataset:** Olist Brazilian E-Commerce Public Dataset (historical data, 2016–2018).

## 1. Load the data

The nine CSV files are stored in Google Drive under `/content/drive/MyDrive/Olist_project/Data/`. The notebook mounts Drive, reads the source files from that folder, and keeps the original imported DataFrames unchanged.

If you start a fresh Colab runtime, run the Drive-mount cell near the top before continuing. Confirm that the path cell reports nine CSV files.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Google Drive project location
PROJECT_DIR = Path("/content/drive/MyDrive/Olist_project")
DATA_DIR = PROJECT_DIR / "Data"  # Folder name in Drive uses capital D
OUTPUT_DIR = PROJECT_DIR / "outputs"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_count = len(list(DATA_DIR.glob("*.csv")))
print("Project folder:", PROJECT_DIR)
print("Data folder:", DATA_DIR)
print("CSV files found:", csv_count)

if csv_count != 9:
    print("Warning: expected 9 CSV files. Check the Data folder contents.")


In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

from pathlib import Path
from IPython.display import display

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style="whitegrid")

# Load datasets from the Google Drive path defined above
files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

data = {}

for name, filename in files.items():
    file_path = DATA_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(f"Missing dataset: {file_path}")
    data[name] = pd.read_csv(file_path)

print("All datasets loaded successfully!")
print("-" * 50)

for name, df in data.items():
    print(f"{name:25s}: {df.shape}")


## 2. Understand data structure and quality

Compare the size and completeness of each table. Then inspect missing values by column. This helps distinguish expected gaps (such as delivery timestamps for orders that were not delivered) from fields that may affect analysis.

In [ ]:

# Dataset overview

overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Cells": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    }
    for name, df in data.items()
])

display(overview)


**Column-level missingness**

The next cell lists only columns containing missing values. Missing review text does not mean the review score is missing, and missing delivery timestamps should be interpreted alongside order status.

In [ ]:

# Missing values by column

for name, df in data.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{name.upper()}")
        display(pd.DataFrame({
            "Missing Count": missing,
            "Missing %": (missing / len(df) * 100).round(2)
        }))


## 3. Prepare analytical data

Create working copies so the original imported DataFrames remain intact. Convert order timestamps, derive purchase month/year and delivery duration/delay, create a late-delivery flag for delivered orders only, and remove exact duplicate geolocation rows from the analytical copy.

In [ ]:

# Preserve original data and create working copies
prepared = {name: df.copy() for name, df in data.items()}

# Convert order timestamps
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    prepared["orders"][col] = pd.to_datetime(
        prepared["orders"][col], errors="coerce"
    )

# Create useful time features
prepared["orders"]["purchase_month"] = (
    prepared["orders"]["order_purchase_timestamp"].dt.to_period("M").astype(str)
)

prepared["orders"]["purchase_year"] = (
    prepared["orders"]["order_purchase_timestamp"].dt.year
)

# Delivery metrics (in days)
prepared["orders"]["delivery_days"] = (
    prepared["orders"]["order_delivered_customer_date"] -
    prepared["orders"]["order_purchase_timestamp"]
).dt.total_seconds() / 86400

prepared["orders"]["delivery_delay_days"] = (
    prepared["orders"]["order_delivered_customer_date"] -
    prepared["orders"]["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Late delivery indicator; meaningful only for delivered orders
prepared["orders"]["is_late"] = pd.Series(
    pd.NA, index=prepared["orders"].index, dtype="Int64"
)

delivered_mask = prepared["orders"]["order_status"].eq("delivered")

prepared["orders"].loc[delivered_mask, "is_late"] = (
    prepared["orders"].loc[delivered_mask, "delivery_delay_days"] > 0
).astype("Int64")

# Translate missing product categories into an explicit label
prepared["products"]["product_category_name"] = (
    prepared["products"]["product_category_name"].fillna("unknown")
)

# Remove exact duplicate geolocation rows in the analytical copy only
prepared["geolocation"] = (
    prepared["geolocation"].drop_duplicates().copy()
)

print("Data preparation completed.")
print("Original datasets remain unchanged.")
print("Geolocation rows after exact duplicate removal:",
      len(prepared["geolocation"]))


**Delivery-field validation**

This check summarises delivery durations and compares late versus on-time/early deliveries. These values are calculated only for orders marked as delivered.

In [ ]:

# Check delivery metrics
print("Delivered orders:", delivered_mask.sum())

display(
    prepared["orders"].loc[
        delivered_mask,
        ["delivery_days", "delivery_delay_days", "is_late"]
    ].describe()
)

print("\nLate vs on-time delivered orders:")
display(
    prepared["orders"].loc[delivered_mask, "is_late"]
    .value_counts(dropna=False)
    .rename(index={0: "On time / early", 1: "Late"})
    .to_frame("Orders")
)


## 4. Create the SQL database

Load the prepared tables into SQLite. Indexes are added to commonly joined or filtered keys to support the business queries.

In [ ]:

# Create SQLite database
DB_PATH = PROJECT_DIR / "olist.db"

conn = sqlite3.connect(DB_PATH)

# Load all nine prepared datasets into SQLite
for name, df in prepared.items():
    df.to_sql(
        name,
        conn,
        if_exists="replace",
        index=False,
        chunksize=500
    )

# Create indexes to improve query performance
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_orders_id ON orders(order_id)",
    "CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)",
    "CREATE INDEX IF NOT EXISTS idx_orders_purchase ON orders(order_purchase_timestamp)",
    "CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(order_id)",
    "CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(product_id)",
    "CREATE INDEX IF NOT EXISTS idx_items_seller ON order_items(seller_id)",
    "CREATE INDEX IF NOT EXISTS idx_customers_id ON customers(customer_id)",
    "CREATE INDEX IF NOT EXISTS idx_products_id ON products(product_id)"
]

for statement in indexes:
    conn.execute(statement)

conn.commit()

print("SQLite database successfully created!")
print("Database:", DB_PATH)


**Database validation**

Confirm that all nine tables exist and that their row counts match the prepared DataFrames.

In [ ]:

# List database tables
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

display(tables)

# Verify row counts
for table in tables["name"]:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total FROM "{table}"',
        conn
    ).iloc[0, 0]

    print(f"{table:25s}: {count:,} rows")


## 5. SQL business analysis

Use SQL joins, aggregations, grouping and a common table expression (CTE) to answer five questions:

1. What is the delivered-order sales value?
2. How did monthly product revenue change over time?
3. Which product categories generated the most revenue?
4. Which sellers generated the most product revenue?
5. How many customers placed more than one delivered order?

**Metric definition:** Product revenue is the sum of item prices. Freight is reported separately. Neither measure represents profit.

### 5.1 Overall delivered-order value

Join orders, customers and order items to calculate delivered orders, distinct customers, product revenue, freight revenue, combined item-plus-freight value and average item price.

In [ ]:

query = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_delivered_orders,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    ROUND(SUM(oi.price), 2) AS product_revenue,
    ROUND(SUM(oi.freight_value), 2) AS freight_revenue,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_order_value,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
"""

overall = pd.read_sql_query(query, conn)
display(overall)


### 5.2 Monthly revenue trend

Aggregate delivered-order item prices by purchase month. The result uses product revenue and excludes freight charges.

In [ ]:

query = """
SELECT
    purchase_month AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY purchase_month
ORDER BY purchase_month;
"""

monthly_sales = pd.read_sql_query(query, conn)
display(monthly_sales)


### 5.3 Top product categories

Translate category names where a translation is available, then rank categories by delivered item revenue. The table shows the ten highest-revenue categories.

In [ ]:

query = """
SELECT
    COALESCE(t.product_category_name_english,
             p.product_category_name,
             'Unknown') AS category,
    COUNT(*) AS items_sold,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation t
    ON p.product_category_name = t.product_category_name
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""

top_categories = pd.read_sql_query(query, conn)
display(top_categories)


### 5.4 Top sellers

Rank sellers by delivered item revenue and show their state, number of distinct orders and items sold. Revenue and sales volume are different measures.

In [ ]:

query = """
SELECT
    oi.seller_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(*) AS items_sold,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN sellers s
    ON oi.seller_id = s.seller_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY oi.seller_id, s.seller_state
ORDER BY revenue DESC
LIMIT 10;
"""

top_sellers = pd.read_sql_query(query, conn)
display(top_sellers)


### 5.5 Repeat-purchase behaviour

Count delivered orders by `customer_unique_id`, which identifies a customer across order-specific customer records. The percentages describe repeat purchases within this dataset's observation window.

In [ ]:

query = """
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)

SELECT
    CASE
        WHEN order_count = 1 THEN 'One-time customer'
        ELSE 'Repeat customer'
    END AS customer_type,
    COUNT(*) AS customers,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_orders),
        2
    ) AS percentage
FROM customer_orders
GROUP BY customer_type;
"""

customer_behavior = pd.read_sql_query(query, conn)
display(customer_behavior)


## 6. Exploratory data analysis

The charts below provide a visual view of the SQL results. Each chart has a short interpretation guide so the reader knows what is being compared and how to read the measure.

### 6.1 Monthly product revenue

Shows the month-by-month value of delivered item sales. The very low values at the beginning of the dataset reflect limited coverage in those early months, so they should not be interpreted as a normal full-month baseline.

In [ ]:

plt.figure(figsize=(14, 5))

plt.plot(
    monthly_sales["month"],
    monthly_sales["revenue"],
    marker="o",
    linewidth=2
)

plt.title("Monthly Revenue Trend", fontsize=16)
plt.xlabel("Month")
plt.ylabel("Revenue (R$)")
plt.xticks(rotation=60)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 6.2 Highest-revenue product categories

Compares the ten categories with the greatest delivered item revenue. This is a revenue ranking, not a ranking by units sold or profit.

In [ ]:

plt.figure(figsize=(11, 6))

sns.barplot(
    data=top_categories,
    x="revenue",
    y="category"
)

plt.title("Top 10 Product Categories by Revenue", fontsize=15)
plt.xlabel("Revenue (R$)")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


### 6.3 One-time versus repeat customers

Shows the share of unique customers with one delivered order versus more than one delivered order during the available observation period.

In [ ]:

plt.figure(figsize=(7, 6))

plt.pie(
    customer_behavior["customers"],
    labels=customer_behavior["customer_type"],
    autopct="%1.1f%%",
    startangle=90
)

plt.title("One-Time vs Repeat Customers")
plt.show()


### 6.4 Highest-revenue sellers

Compares the ten sellers with the greatest delivered item revenue. Seller IDs are shortened to their first eight characters for chart readability; the full IDs remain available in the SQL result table.

In [ ]:

plt.figure(figsize=(11, 6))

seller_plot = top_sellers.copy()
seller_plot["seller_label"] = seller_plot["seller_id"].str[:8]

sns.barplot(
    data=seller_plot,
    x="revenue",
    y="seller_label"
)

plt.title("Top 10 Sellers by Revenue")
plt.xlabel("Revenue (R$)")
plt.ylabel("Seller ID")
plt.tight_layout()
plt.show()


## 7. Customer segmentation (RFM)

RFM summarises customer behaviour using:

- **Recency:** days since the customer's most recent delivered purchase (lower is more recent).
- **Frequency:** number of delivered orders.
- **Monetary:** total item spend across delivered orders (freight excluded).

The next cell constructs one row per `customer_unique_id`. Scores and segment labels use analyst-defined thresholds for exploratory grouping; they are not a validated churn model.

In [ ]:
# Build one RFM record per unique customer using delivered orders only
delivered_orders = prepared["orders"].loc[
    prepared["orders"]["order_status"].eq("delivered"),
    ["order_id", "customer_id", "order_purchase_timestamp"]
].copy()

customer_lookup = prepared["customers"][["customer_id", "customer_unique_id"]]
delivered_customer_orders = delivered_orders.merge(
    customer_lookup, on="customer_id", how="left"
)

# Item spend per order, excluding freight
order_spend = (
    prepared["order_items"]
    .groupby("order_id", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "order_spend"})
)
delivered_customer_orders = delivered_customer_orders.merge(
    order_spend, on="order_id", how="left"
)

reference_date = (
    delivered_customer_orders["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

customer_spending = (
    delivered_customer_orders
    .groupby("customer_unique_id")
    .agg(
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("order_spend", "sum")
    )
    .reset_index()
)
customer_spending["recency"] = (
    reference_date - customer_spending["last_purchase"]
).dt.days

print("RFM reference date:", reference_date.date())
print("Unique customers included:", f"{len(customer_spending):,}")
display(customer_spending.head())

### 7.1 Assign RFM scores and customer segments

Recency scores range from 1 (least recent) to 5 (most recent). Frequency uses fixed purchase-count bands to avoid splitting the large group of one-time customers into artificial quintiles. Monetary scores are quintiles of customer spend.

The segment labels are simple rules based on recency and frequency; monetary score is retained for analysis but does not determine the segment label.

In [ ]:

# Corrected RFM scoring

# Recency: lower days = better score
customer_spending["R_score"] = pd.cut(
    customer_spending["recency"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: score based on actual purchase count
customer_spending["F_score"] = pd.cut(
    customer_spending["frequency"],
    bins=[0, 1, 2, 4, 8, np.inf],
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: rank customers by spending
customer_spending["M_score"] = pd.qcut(
    customer_spending["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Revised segments
def segment_customer(row):
    if row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Loyal / Active"
    elif row["R_score"] >= 4 and row["F_score"] == 1:
        return "Recent One-Time"
    elif row["R_score"] <= 2 and row["F_score"] >= 2:
        return "At Risk"
    elif row["R_score"] <= 2:
        return "Inactive One-Time"
    else:
        return "Regular Customers"

customer_spending["segment"] = customer_spending.apply(
    segment_customer, axis=1
)

rfm_summary = (
    customer_spending.groupby("segment", observed=True)
    .agg(
        customers=("customer_unique_id", "count"),
        avg_frequency=("frequency", "mean"),
        avg_spending=("monetary", "mean")
    )
    .round(2)
)

display(rfm_summary)


### 7.2 Compare segment sizes

This chart shows the number of customers assigned to each RFM segment. Read the counts alongside the segment summary table above; a large segment does not necessarily have high average spend.

In [ ]:

segment_summary = customer_spending["segment"].value_counts()

plt.figure(figsize=(9, 5))

sns.barplot(
    x=segment_summary.values,
    y=segment_summary.index
)

plt.title("Customer Segmentation using RFM")
plt.xlabel("Number of Customers")
plt.ylabel("Customer Segment")
plt.tight_layout()
plt.show()


## 8. Delivery performance

Evaluate delivery duration and delivery delay for delivered orders only.

- **Delivery time:** purchase timestamp to customer delivery timestamp.
- **Delivery delay:** actual delivery date minus estimated delivery date.
- A negative delay means the order arrived before the estimate; a positive delay means it arrived late.

In [ ]:

delivery_data = prepared["orders"].query(
    "order_status == 'delivered'"
).copy()

delivery_summary = pd.DataFrame({
    "Metric": [
        "Average delivery time (days)",
        "Median delivery time (days)",
        "Average delivery delay (days)",
        "Late delivery percentage"
    ],
    "Value": [
        delivery_data["delivery_days"].mean(),
        delivery_data["delivery_days"].median(),
        delivery_data["delivery_delay_days"].mean(),
        delivery_data["is_late"].mean() * 100
    ]
})

display(delivery_summary.round(2))


### 8.1 Distribution of delivery delays

The vertical reference at zero separates early deliveries (left) from late deliveries (right). This chart shows the distribution, while the summary table above provides the average, median and late-delivery percentage.

In [ ]:

plt.figure(figsize=(10, 5))

sns.histplot(
    delivery_data["delivery_delay_days"].dropna(),
    bins=50,
    kde=True
)

plt.axvline(0, linestyle="--", label="Estimated delivery date")
plt.title("Distribution of Delivery Delays")
plt.xlabel("Days (Positive = Late, Negative = Early)")
plt.ylabel("Number of Orders")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Business performance dashboard

The dashboard combines four views: monthly product revenue, leading product categories, RFM customer segment sizes and delivery-delay distribution. It is saved as `olist_dashboard.png` in the main project folder.

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(17, 11))

# 1. Monthly revenue
axes[0, 0].plot(
    monthly_sales["month"],
    monthly_sales["revenue"],
    marker="o"
)
axes[0, 0].set_title("Monthly Revenue Trend")
axes[0, 0].set_ylabel("Revenue (R$)")
axes[0, 0].tick_params(axis="x", rotation=60)
axes[0, 0].grid(alpha=0.3)

# 2. Top categories
sns.barplot(
    data=top_categories.head(7),
    x="revenue",
    y="category",
    ax=axes[0, 1]
)
axes[0, 1].set_title("Top Product Categories by Revenue")
axes[0, 1].set_xlabel("Revenue (R$)")

# 3. Customer segments
segment_data = (
    customer_spending["segment"]
    .value_counts()
    .sort_values(ascending=True)
)

axes[1, 0].barh(
    segment_data.index,
    segment_data.values
)
axes[1, 0].set_title("Customer Segmentation (RFM)")
axes[1, 0].set_xlabel("Number of Customers")

# 4. Delivery performance
sns.histplot(
    delivery_data["delivery_delay_days"].dropna(),
    bins=40,
    ax=axes[1, 1]
)
axes[1, 1].axvline(
    0, linestyle="--", label="Estimated delivery date"
)
axes[1, 1].set_title("Delivery Delay Distribution")
axes[1, 1].set_xlabel("Delay in Days")
axes[1, 1].legend()

plt.suptitle(
    "OLIST E-COMMERCE BUSINESS ANALYSIS",
    fontsize=19,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(
    PROJECT_DIR / "olist_dashboard.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 10. Executive summary

### Key findings

- **Sales:** Delivered orders generated approximately **R$13.22 million** in product revenue, excluding freight. November 2017 was the highest-revenue month in the observed period.
- **Products:** Health & Beauty led the categories by product revenue.
- **Customers:** Approximately **97%** of delivered customers placed one order in the observation window. This indicates low observed repeat purchasing, not necessarily permanent churn.
- **Sellers:** The highest-revenue seller generated approximately **R$226,988** in delivered item sales.
- **Delivery:** Average delivery time was **12.56 days**; **8.11%** of delivered orders arrived after the estimated date. Average delay was **−11.18 days**, meaning deliveries were earlier than estimated on average.

### Recommendations

1. **Encourage second purchases:** Test targeted follow-up campaigns for one-time customers and re-engagement for customers with longer recency.
2. **Use category insights:** Explore merchandising and campaign opportunities in high-revenue categories, while checking margin and inventory data before making investment decisions.
3. **Learn from strong sellers:** Compare high-performing sellers on order volume, product mix and delivery performance before generalising their practices.
4. **Investigate late deliveries:** Break late-delivery rates down by seller, region and product characteristics to identify operational drivers.
5. **Build customer value:** Test loyalty or cross-category initiatives and measure whether they increase repeat purchase rates.

### Conclusion

The analysis combines relational SQL queries with Python-based visualisation and customer segmentation to describe historical marketplace performance and identify areas for further investigation.

## 11. Limitations and interpretation notes

- The dataset covers historical transactions from 2016–2018 and may not represent current marketplace behaviour.
- Product revenue and freight charges are sales-value measures, not profit; cost and margin data are unavailable.
- Repeat-purchase analysis is constrained by the dataset's observation window. A customer with one recorded order may have purchased outside that period.
- RFM thresholds and segment names are analyst-defined exploratory rules, not validated churn predictions.
- Missing timestamps and product attributes limit some analyses. Missing review text does not imply a missing review score.
- Seller revenue alone does not measure seller profitability or service quality.
- Early months have limited transaction coverage and should be interpreted cautiously in trend comparisons.

## 12. Export project outputs

Save the principal analytical tables to `outputs/`. The dashboard is saved separately in the project root by the dashboard cell above. The SQLite connection is closed after exporting.

In [ ]:

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Save analytical results
monthly_sales.to_csv(OUTPUT_DIR / "monthly_sales.csv", index=False)
top_categories.to_csv(OUTPUT_DIR / "top_categories.csv", index=False)
top_sellers.to_csv(OUTPUT_DIR / "top_sellers.csv", index=False)
customer_behavior.to_csv(OUTPUT_DIR / "customer_behavior.csv", index=False)
rfm_summary.to_csv(OUTPUT_DIR / "rfm_summary.csv")
delivery_summary.to_csv(OUTPUT_DIR / "delivery_summary.csv", index=False)

# Save customer-level RFM data
customer_spending.to_csv(
    OUTPUT_DIR / "customer_rfm.csv",
    index=False
)

# Close database connection
conn.close()

print("Project outputs saved successfully!")
print("Output directory:", OUTPUT_DIR)
print("\nFiles:")
for file in OUTPUT_DIR.iterdir():
    print(file.name)
